# Cuaderno para realizar el cargue de la data y visualizacion inicial del dataset

## Diccionario de Datos

| No | Variable_español | Variable | Descripción |
| :--- | :--- | :--- | :---|
| 1 | Identificador | **UID** | Identificador único correlativo desde 1 hasta 10,000. |
| 2 | ID productor | **product ID** | ID de variante de calidad; número de serie específico que incluye la letra de la variante. |
| 3 | Tipo de producto | **type** | Tipo de producto basado en el ID: **L** (Bajo - 50%), **M** (Medio - 30%) o **H** (Alto - 20%). |
| 4 | Temperatura del aire | **air temperature [K]** | Temperatura del aire en Grados Kelvin (K); generada mediante *random walk* y normalizada a 300 K con una desviación estándar de 2 K. |
| 5 | Temperatura del proceso de fabricacion | **process temperature [K]** | Temperatura del proceso de manufactura en escala Kelvin [K]. |
| 6 | Velocidad de rotacion (RPM) | **rotational speed [rpm]** | Velocidad de rotación en rpm; calculada a partir de una potencia de 2860 W con ruido de distribución normal. |
| 7 | Torque | **torque [Nm]** | Par motor (torque) en Nm; distribuido normalmente alrededor de 40 Nm (DE = 10 Nm) sin valores negativos. |
| 8 | Desgaste | **tool wear [min]** | Desgaste de la herramienta; las variantes H/M/L añaden 5/3/2 minutos de desgaste respectivamente al proceso. |
| 9 | Fallo de maquina | **machine failure** | **Variable objetivo:** Indica si la máquina falló (1) o no falló (0) durante el proceso. |

## Se enlistas las posibles fallas que pueda tener estas herramientas

| No. | Tipo de falla | Descripción |
| :--- | :--- | :--- |
| **1** | **Falla por desgaste de herramienta (TWF)** | La herramienta se reemplaza si falla en un tiempo de uso seleccionado al azar entre 200 y 240 minutos. En estos puntos, la herramienta fue reemplazada 69 veces y falló en 51 ocasiones. |
| **2** | **Falla por disipación de calor (HDF)** | La disipación de calor causa una falla en el proceso si la diferencia entre la temperatura del aire y la del proceso es inferior a 8,6 K y la velocidad de rotación es menor a 1.380 rpm. |
| **3** | **Falla de potencia (PWF)** | Si la potencia (torque x velocidad) es inferior a 3.500 W o superior a 9.000 W, el proceso falla. Esto sucede 95 veces en el conjunto de datos. |
| **4** | **Falla por sobreesfuerzo (OSF)** | Si el producto del desgaste de la herramienta y el torque supera los 11.000 minNm (L), 12.000 (M) o 13.000 (H), el proceso falla por sobrecarga. |
| **5** | **Fallas aleatorias (RNF)** | Cada proceso tiene una probabilidad del 0,1% de fallar independientemente de sus parámetros. Ocurrió en 5 puntos de datos. |

TWF

Español: FDH — Falla por Desgaste de Herramienta

Inglés: Tool Wear Failure

HDF

Español: FDC — Falla por Disipación de Calor

Inglés: Heat Dissipation Failure

PWF

Español: FP — Falla de Potencia

Inglés: Power Failure

OSF

Español: FS — Falla por Sobreesfuerzo (o Sobrecarga)

Inglés: Overstrain Failure

RNF

Español: FA — Fallas Aleatorias

Inglés: Random Failures

## 1. Importacion de las libreas necesarias para realizar el EDA, estas librerias ya fueron instaladas en el entorno virtual

In [ ]:
import pandas as pd
import numpy as np

Configuración para ver todas las columnas en el cuaderno

In [2]:
pd.set_option('display.max_columns', None)

Definir la ruta al dataset (Data Raw)
'../' sale de la carpeta notebooks, luego entra a data/raw/

In [3]:
ruta_data = "../data/raw/ai4i2020.csv"

In [4]:
# Cargar el dataset
df = pd.read_csv(ruta_data)

# Mostrar las primeras 5 filas para confirmar la carga
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
df.shape

(10000, 14)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
UDI,10000.0,5000.50000,2886.895680,1.0,2500.75,5000.5,7500.25,10000.0
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.30,300.1,301.50,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.80,310.1,311.10,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.00,1503.0,1612.00,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.20,40.1,46.80,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.00,108.0,162.00,253.0
Machine failure,10000.0,0.03390,0.180981,0.0,0.00,0.0,0.00,1.0
TWF,10000.0,0.00460,0.067671,0.0,0.00,0.0,0.00,1.0
HDF,10000.0,0.01150,0.106625,0.0,0.00,0.0,0.00,1.0
PWF,10000.0,0.00950,0.097009,0.0,0.00,0.0,0.00,1.0


In [8]:
df["Machine failure"].value_counts()

Machine failure
0    9661
1     339
Name: count, dtype: int64

In [9]:
print(df.columns)

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='str')


In [10]:
df.isnull().sum()

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [11]:
df["Type"].value_counts()

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64